# 🖥️ Delft3D tutorial

This tutorial shows how to load in [Delft3D](https://www.deltares.nl/en/software-and-data/products/delft3d-4-suite) model output files (in NetCDF format) into Parcels. It treats both the [unstructured grid format (Delft3D-fm)](#unstructured-grids-delft3d-fm) and the [structured grid format (Delft3D-flow)](#structured-grids-delft3d-flow).

In [ ]:
import cmocean as cmo
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import pandas as pd
import uxarray as ux
import xarray as xr
from matplotlib.collections import PolyCollection

import parcels
import parcels.tutorial

## Unstructured Grids (Delft3D-fm)

Delft3D-fm is the flexible mesh version of Delft3D. It uses an unstructured grid, which is supported in Parcels v4. 

```{note}
There is one major caveat to using Delft3D-fm with Parcels: currently, Parcels v4 can only handle triangular meshes, while Delft3D-fm can also use quadrilateral meshes. Therefore, if you want to use Parcels v4 with Delft3D-fm, you need to first modify your model output to be based on a triangular mesh. Any quadrilateral grid cell can be split into two triangular grid cells, for example by drawing a diagonal line between two opposite corners of the quadrilateral. However, the Parcels results will only be independent of this regridding if you use `UxConstantFace...` interpolation.
```

The following example shows how to load in a Delft3D-fm model output file and create a `FieldSet` from it. 

We first show the raw file that is a combination of triangular and quadrilateral grid cells. This is a small region of the North Sea, just offshore of the city of The Hague.

In [ ]:
raw_ds = parcels.tutorial.open_dataset("Delft3D_data/delft3d-fm-raw")

node_x = raw_ds["Mesh_node_x"].values
node_y = raw_ds["Mesh_node_y"].values
raw_face_nodes = raw_ds["Mesh_face_nodes"].values

raw_polygons = [
    [(node_x[i], node_y[i]) for i in face[np.isfinite(face)].astype(np.int64) - 1]
    for face in raw_face_nodes
]

fig, ax = plt.subplots(figsize=(5, 3))
ax.add_collection(
    PolyCollection(raw_polygons, edgecolor="k", facecolor="none", linewidth=0.3)
)
ax.set_aspect("equal")

We now show how to create a regridded version that only contains triangular grid cells.

In [ ]:
# Use integer-safe connectivity for triangulation logic (1-based with 0 as invalid).
raw_face_nodes_i = np.where(np.isfinite(raw_face_nodes), raw_face_nodes, 0).astype(
    "int64"
)

tri_list = []
tri_parent_face = []

for i, face in enumerate(raw_face_nodes_i):
    # UGRID connectivity is typically 1-based with 0/-1 as fill values.
    valid = face[face > 0] - 1
    n_valid = valid.size

    if n_valid < 3:
        continue
    if n_valid == 3:
        tri_list.append(valid)
        tri_parent_face.append(i)
        continue

    if n_valid == 4:
        a, b, c, d = valid
        diag_ac = (node_x[a] - node_x[c]) ** 2 + (node_y[a] - node_y[c]) ** 2
        diag_bd = (node_x[b] - node_x[d]) ** 2 + (node_y[b] - node_y[d]) ** 2

        if diag_ac <= diag_bd:
            tri_list.extend([[a, b, c], [a, c, d]])
        else:
            tri_list.extend([[a, b, d], [b, c, d]])
        tri_parent_face.extend([i, i])
        continue

    # Fallback for polygons with >4 nodes: simple fan triangulation.
    for j in range(1, n_valid - 1):
        tri_list.append([valid[0], valid[j], valid[j + 1]])
        tri_parent_face.append(i)

tri_faces = np.asarray(tri_list, dtype="int64")
tri_parent_face = np.asarray(tri_parent_face, dtype="int64")

zf = [0, 1]
zc = [0.5]

uxgrid = ux.Grid.from_topology(
    node_lon=raw_ds["Mesh_node_x"],
    node_lat=raw_ds["Mesh_node_y"],
    face_node_connectivity=tri_faces,
    fill_value=-1,
)

# Fast remap from original faces -> triangular faces using NumPy indexing.
tri_parent_idx = tri_parent_face.astype(np.intp, copy=False)


def to_face_3d(ds, var_name, target_name):
    src = ds[var_name].transpose("time", "nMesh_face").astype("float32")
    tri_vals = np.take(src.values, tri_parent_idx, axis=1)
    return xr.DataArray(
        tri_vals[:, np.newaxis, :],
        dims=("time", "zc", "n_face"),
        coords={"time": ds["time"], "zc": zc},
        name=target_name,
        attrs=src.attrs,
    )


uxds = ux.UxDataset(
    xr.Dataset(
        {
            "U": to_face_3d(raw_ds, "velu", "U"),
            "V": to_face_3d(raw_ds, "velv", "V"),
        },
        coords={"zf": ("zf", zf), "zc": ("zc", zc), "time": raw_ds["time"]},
    ),
    uxgrid=uxgrid,
)
fieldset = parcels.FieldSet.from_ugrid_conventions(uxds, mesh="spherical")
fieldset.describe()

We can also visualise this regridded version of the model output, which is now compatible with Parcels v4. As you can see, all the quadrilateral grid cells have been split into two triangular grid cells.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
triang = mtri.Triangulation(
    uxgrid["node_lon"].data,
    uxgrid["node_lat"].data,
    triangles=uxgrid["face_node_connectivity"].data,
)

ax.triplot(triang, color="k", lw=0.3, alpha=0.5, label="Flow mesh", zorder=2)
ax.set_aspect("equal")

And can do a simple Parcels simulation with the regridded version of the model output.

In [ ]:
X, Y = np.meshgrid(np.linspace(4.02, 4.18, 3), np.linspace(52.05, 52.15, 3))
pset = parcels.ParticleSet(fieldset, x=X, y=Y)

outfile = parcels.ParticleFile(
    "tutorial_delft3d_particles.parquet", outputdt=np.timedelta64(15, "m"), mode="w"
)

pset.execute(
    parcels.kernels.AdvectionRK2,
    dt=np.timedelta64(60, "s"),
    endtime=fieldset.time_interval.right,
    output_file=outfile,
)

And then we can plot the trajectories of the particles on top of the flow mesh.

In [ ]:
df = parcels.read_particlefile(outfile._path)
fig, ax = plt.subplots(figsize=(5, 3))
ax.triplot(triang, color="k", lw=0.3, alpha=0.5, label="Flow mesh", zorder=2)
ax.set_aspect("equal")
for traj in df.partition_by("particle_id", maintain_order=True):
    ax.plot(traj["x"], traj["y"], "b.-")
plt.show()

## Structured Grids (Delft3D-flow)
Special about Delft3D is that its structured grid contains NaN values for points that don't exist in the model domain. The hashtable approach in Parcels v4 supports these NaN grid cells out of the box, so that you can use the model output directly without any preprocessing.

The example below is of a small domain in the Port of Rotterdam. We first pick the coorindates and the velocity fields from the dataset, and then convert them to the sgrid conventions. Finally, we create a FieldSet from the converted dataset.

```{note}
While Delft3D provides velocities on a CGrid, we currently can't use the {py:obj}`parcels.interpolators.CGrid_Velocity` interpolator on the velocity fields. This is because the velocities have been rotated to east and north in the output - but the Parcels interpolator expects them in the `M` and `N` (along-grid) directions. Therefore, we use the {py:obj}`parcels.interpolators.XFreeslip` interpolator instead.
```

In [ ]:
ds = parcels.tutorial.open_dataset("Delft3D_data/Rotterdam_tiny")
coords = ds[["XZETA", "YZETA", "SIGMA_C"]]
ds_fset = parcels.convert.delft3d_to_sgrid(
    fields={"U": ds["VELU"], "V": ds["VELV"]}, coords=coords
)
fieldset = parcels.FieldSet.from_sgrid_conventions(ds_fset)

# Set the interpolation method for the UV field to XFreeslip (see note above)
fieldset.UV.interp_method = parcels.interpolators.XFreeslip()

fieldset = fieldset.to_windowed_arrays()
fieldset.describe()

Now we define a grid of a few particles to release in the domain, and run a simple advection simulation. The particles are advected by the Delft3D velocity fields, and we can visualize their trajectories.

In [ ]:
# Define a set of starting points
points_x, points_y = np.meshgrid(
    np.linspace(93000, 93300, 5), np.linspace(436300, 436600, 5)
)
z = np.full(points_x.shape, 0.06)  # Depth of 0.06 m
pset = parcels.ParticleSet(fieldset, x=points_x, y=points_y, z=z)

# St up an output file to save particle trajectories
outputfile = parcels.ParticleFile(
    "Delft3D_structured.parquet",
    outputdt=np.timedelta64(60, "s"),
    mode="w",
)


# Define a custom error handling kernel to delete particles on any error
def DeleteOnAnyError(particles, fieldset):
    any_error = particles.state >= 50  # This captures all Errors
    particles[any_error].state = parcels.StatusCode.Delete


# Run the particle set with the advection kernel and the custom error handling kernel
pset.execute(
    [parcels.kernels.AdvectionRK2, DeleteOnAnyError],
    endtime=fieldset.time_interval.right,
    dt=np.timedelta64(60, "s"),
    output_file=outputfile,
    verbose_progress=False,
)

In [ ]:
df = parcels.read_particlefile("Delft3D_structured.parquet")

fig, ax = plt.subplots(figsize=(8, 6))

# Plot background velocity field
time_idx = 0
layer_idx = 0
speed = (
    np.sqrt(ds["VELV"] ** 2 + ds["VELU"] ** 2)
    .isel(TIME=time_idx, LAYER=layer_idx)
    .compute()
)
tpc = plt.pcolor(
    speed.XZETA.values,
    speed.YZETA.values,
    speed,
    edgecolors="black",
    cmap="cmo.speed",
    vmin=0,
    vmax=np.max(np.abs(speed)),
)
time_str = pd.to_datetime(speed.TIME.values).strftime("%Y-%m-%d %H:%M:%S")
fig.colorbar(
    tpc,
    ax=ax,
    label=f"Flow speed on {time_str} and {speed.SIGMA_C.values:.2f}m depth [m/s]",
)

for traj in df.partition_by("particle_id"):
    ax.plot(traj["x"], traj["y"], "b", alpha=0.8)

ax.plot(points_x, points_y, "m.", label="initial particle positions")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel(ds.XZETA.long_name)
ax.set_ylabel(ds.YZETA.long_name)
plt.show()